In [1]:
import pandas as pd
import json
import requests
from pydantic import BaseModel, Field, field_validator


# ===== CONFIGURAÇÃO DO LOADER =====
class DataLoaderConfig(BaseModel):
    url: str = Field(..., description="URL do arquivo JSON")
    chave: str | None = Field(None, description="Chave do JSON a ser carregada")

    @field_validator("url")
    @classmethod
    def validar_url(cls, v: str) -> str:
        if not (v.startswith("http://") or v.startswith("https://")):
            raise ValueError("URL deve começar com http:// ou https://")
        return v

    model_config = {
        "extra": "allow",
        "json_schema_extra": {
            "example": {
                "url": "https://exemplo.com/dados.json",
                "chave": "dados_churn",
            }
        },
    }

# ===== FUNÇÃO DE CARREGAMENTO =====
def load_data(config: DataLoaderConfig) -> pd.DataFrame:
    """
    Carrega um JSON da URL e transforma em DataFrame.
    Se houver uma chave, extrai os dados dessa chave.
    """
    response = requests.get(config.url)
    response.raise_for_status()
    data = response.json()

    # Normaliza caso seja dict com chave
    if isinstance(data, dict) and config.chave:
        data = data[config.chave]

    # Caso seja lista ou dict, transforma em DataFrame
    if isinstance(data, list):
        dados_normalizados = pd.json_normalize(data)
    elif isinstance(data, dict):
       dados_normalizados = pd.json_normalize([data])
    else:
        dados_normalizados = pd.DataFrame([data])

    return dados_normalizados

# ===== EXEMPLO DE USO =====
configs = {
    "churn": DataLoaderConfig(
        url="https://raw.githubusercontent.com/YuriArduino/Estudos_Pandas/refs/heads/data-tests/dataset-telecon.json",
        chave="dados_telecon"  # ajuste conforme a chave do JSON
    )
}

# Carregar e exibir
dados_normalizados = load_data(configs["churn"])
print(dados_normalizados.head())
print(f"Shape do DataFrame: {dados_normalizados.shape}")

   id_cliente Churn cliente.genero  cliente.idoso cliente.parceiro  \
0  0002-ORFBO   nao       feminino              0              sim   
1  0003-MKNFE   nao      masculino              0              nao   
2  0004-TLHLJ   sim      masculino              0              nao   
3  0011-IGKFF   sim      masculino              1              sim   
4  0013-EXCHZ   sim       feminino              1              sim   

  cliente.dependentes  cliente.tempo_servico telefone.servico_telefone  \
0                 sim                    9.0                       sim   
1                 nao                    9.0                       sim   
2                 nao                    4.0                       sim   
3                 nao                   13.0                       sim   
4                 nao                    3.0                       sim   

  telefone.varias_linhas internet.servico_internet  ...  \
0                    nao                       DSL  ...   
1               

#Entendendo os dados

##Vamos dar atenção para as três colunas abaixo:

*  Na seção "Cliente", a coluna tempo_servico (na tabela, cliente.tempo_servico) que trata dos meses de contrato da pessoa cliente.
*  Na seção "Conta", a coluna cobranca.mensal (na tabela, conta.cobranca.mensal) que trata do valor mensal que a pessoa cliente deve pagar
*  Ainda na seção "Conta", a coluna cobranca.total (na tabela, conta.cobranca.total) que trata do valor total gasto pela pessoa cliente
Se multiplicarmos cobranca.mensal por tempo_servico, teremos o valor de cobranca.total.

Retiraremos informações iniciais dos nossos dados. Para isso, utilizaremos o método info do Pandas.




In [2]:
dados_normalizados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7344 entries, 0 to 7343
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id_cliente                     7344 non-null   object 
 1   Churn                          7344 non-null   object 
 2   cliente.genero                 7344 non-null   object 
 3   cliente.idoso                  7344 non-null   int64  
 4   cliente.parceiro               7344 non-null   object 
 5   cliente.dependentes            7344 non-null   object 
 6   cliente.tempo_servico          7336 non-null   float64
 7   telefone.servico_telefone      7344 non-null   object 
 8   telefone.varias_linhas         7344 non-null   object 
 9   internet.servico_internet      7344 non-null   object 
 10  internet.seguranca_online      7344 non-null   object 
 11  internet.backup_online         7344 non-null   object 
 12  internet.protecao_dispositivo  7344 non-null   o

Temos 7344 entradas que vão de 0 à 7343. Esta informação nos diz que não há nenhum índice saltado.

Se tivéssemos 7344 entradas que vão de 0 à 7500, por exemplo, teríamos um problema de índices na base de dados.

Além disso, temos o total de 21 colunas.

Na tabela propriamente dita, vemos quatro colunas com os seguintes dados:

*  #, que possui o índice
*  Column, com o nome da nossa coluna
*  Non-Null Count, com a quantidade de valores não nulos
*  DType, com o tipo do dado

O que devemos fazer quando temos colunas com tipos incorretos no banco de dados — principalmente quando pretendemos inserir estes dados em um modelo de Machine Learning? Transformá-los na tipagem correta por meio de um cast, permitindo que este modelo possa tratá-los de forma correta e gerar resultados assertivos.

##Identificando valores estranhos

`cast`

Para realizar o cast, acessaremos a célula vazia localizada embaixo da tabela que foi retornada por nosso último comando.

Nela, escreveremos o nome do campo de dados (dados_normalizados[]), que terá entre colchetes e aspas simples o nome da coluna a ser corrigida (conta.cobranca.Total).

In [3]:
dados_normalizados['conta.cobranca.Total'] = dados_normalizados['conta.cobranca.Total'].astype(float)

ValueError: could not convert string to float: ' '

Erro: Isto significa que temos uma string estranha nesta coluna. Por este motivo, ela adquiriu o tipo object.

Temos que tratar este valor estranho da coluna antes de realizar o cast. Faremos isso a seguir.


### Para saber mais: Dicionário dos dados


Um **dicionário de dados** é uma ferramenta essencial para a gestão de informações em uma empresa. Ele descreve cada dado armazenado, ajudando a garantir **consistência e precisão**, e facilita que a equipe de análise compreenda melhor as informações disponíveis, apoiando decisões mais informadas.

Abaixo, o dicionário dos dados utilizados durante o curso:

#### Cliente

* **genero**: gênero do(a) cliente (masculino ou feminino)
* **idoso**: indica se o(a) cliente tem 65 anos ou mais
* **parceiro**: se possui parceiro(a) ou não
* **dependentes**: se possui dependentes ou não
* **tempo\_servico**: tempo de contrato em meses

#### Serviço de telefonia

* **servico\_telefone**: assinatura do serviço telefônico
* **varias\_linhas**: se possui mais de uma linha

#### Serviço de internet

* **servico\_internet**: assinatura do provedor de internet
* **seguranca\_online**: assinatura adicional de segurança online
* **backup\_online**: assinatura adicional de backup online
* **protecao\_dispositivo**: proteção extra para dispositivos
* **suporte\_tecnico**: assinatura de suporte técnico com menor tempo de espera
* **tv\_streaming**: assinatura de TV a cabo
* **filmes\_streaming**: assinatura de streaming de filmes

#### Conta

* **contrato**: tipo de contrato do(a) cliente
* **faturamento\_eletronico**: preferência por receber faturas online
* **metodo\_pagamento**: forma de pagamento utilizada
* **cobranca.mensal**: total mensal de todos os serviços do(a) cliente
* **cobranca.Total**: total gasto pelo(a) cliente

---

##Modificando o tipo da coluna

###Tratando os valores da coluna

In [4]:
dados_normalizados[dados_normalizados['conta.cobranca.Total'] == ' '].head()

,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
975,1371-DWPAZ,nao,feminino,0,sim,sim,0.0,nao,sem servico de telefone,DSL,...,sim,sim,sim,sim,nao,dois anos,nao,cartao de credito (automatico),56.05,
1775,2520-SGTTA,nao,feminino,0,sim,sim,0.0,sim,nao,nao,...,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,dois anos,nao,cheque pelo correio,20.00,
1955,2775-SEFEE,nao,masculino,0,nao,sim,0.0,sim,sim,DSL,...,sim,nao,sim,nao,nao,dois anos,sim,transferencia bancaria (automatica),61.90,
2075,2923-ARZLG,nao,masculino,0,sim,sim,0.0,sim,nao,nao,...,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,dois anos,sim,cheque pelo correio,19.70,
2232,3115-CZMZD,nao,masculino,0,nao,sim,0.0,sim,nao,nao,...,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,dois anos,nao,cheque pelo correio,20.25,


In [5]:
#Lista de colunas a serem referencias
dados_normalizados[dados_normalizados['conta.cobranca.Total'] == ' '][
    ['cliente.tempo_servico', 'conta.contrato',
     'conta.cobranca.mensal', 'conta.cobranca.Total']
]

,cliente.tempo_servico,conta.contrato,conta.cobranca.mensal,conta.cobranca.Total
975,0.0,dois anos,56.05,
1775,0.0,dois anos,20.00,
1955,0.0,dois anos,61.90,
2075,0.0,dois anos,19.70,
2232,0.0,dois anos,20.25,
2308,0.0,dois anos,25.35,
2930,0.0,dois anos,73.35,
3134,0.0,dois anos,25.75,
3203,0.0,dois anos,52.55,
4169,0.0,dois anos,80.85,


Se em todos os casos o tempo de contrato é dois anos, o tempo de serviço que a pessoa cliente terá ao final deste período serão 24 meses. A partir dessa descoberta, basta multiplicar cliente.tempo_servico (24 meses) por conta.cobranca.mensal para inferir o valor de conta.cobranca.Total.

In [6]:
dados_normalizados[dados_normalizados['conta.cobranca.Total'] == ' '].index

Index([975, 1775, 1955, 2075, 2232, 2308, 2930, 3134, 3203, 4169, 5599], dtype='int64')

In [7]:
idx = dados_normalizados[dados_normalizados['conta.cobranca.Total'] == ' '].index

In [8]:
dados_normalizados.loc[idx, "conta.cobranca.Total"] = dados_normalizados.loc[idx, "conta.cobranca.mensal"] * 24

In [9]:
dados_normalizados.loc[idx, "cliente.tempo_servico"] = 24

In [10]:
dados_normalizados.loc[idx][
['cliente.tempo_servico', 'conta.contrato', 'conta.cobranca.mensal', 'conta.cobranca.Total']
]

,cliente.tempo_servico,conta.contrato,conta.cobranca.mensal,conta.cobranca.Total
975,24.0,dois anos,56.05,1345.2
1775,24.0,dois anos,20.00,480.0
1955,24.0,dois anos,61.90,1485.6
2075,24.0,dois anos,19.70,472.8
2232,24.0,dois anos,20.25,486.0
2308,24.0,dois anos,25.35,608.4
2930,24.0,dois anos,73.35,1760.4
3134,24.0,dois anos,25.75,618.0
3203,24.0,dois anos,52.55,1261.2
4169,24.0,dois anos,80.85,1940.4


In [11]:
#Agora podemos usar o cast e converter para float
dados_normalizados['conta.cobranca.Total'] = dados_normalizados['conta.cobranca.Total'].astype(float)
dados_normalizados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7344 entries, 0 to 7343
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id_cliente                     7344 non-null   object 
 1   Churn                          7344 non-null   object 
 2   cliente.genero                 7344 non-null   object 
 3   cliente.idoso                  7344 non-null   int64  
 4   cliente.parceiro               7344 non-null   object 
 5   cliente.dependentes            7344 non-null   object 
 6   cliente.tempo_servico          7336 non-null   float64
 7   telefone.servico_telefone      7344 non-null   object 
 8   telefone.varias_linhas         7344 non-null   object 
 9   internet.servico_internet      7344 non-null   object 
 10  internet.seguranca_online      7344 non-null   object 
 11  internet.backup_online         7344 non-null   object 
 12  internet.protecao_dispositivo  7344 non-null   o

###Para saber mais: Por que alterar o tipo da coluna?



Fazer o cast das colunas para os tipos corretos é fundamental antes de inserir os dados em um modelo de machine learning. Os principais motivos são:

1)  Precisão dos resultados
Garantir que os dados estejam no tipo correto evita perda de informação. Por exemplo, representar números decimais como inteiros causa perda de precisão, impactando diretamente a acurácia do modelo.

2)  Velocidade de processamento
Dados no tipo correto são processados mais rapidamente. Usar, por exemplo, float32 em vez de float64 reduz cálculos e consumo de memória, agilizando operações e treinamentos.

3)   Compatibilidade com o modelo
Alguns modelos exigem tipos específicos de entrada. Ajustar as colunas garante que o modelo aceite os dados sem erros.

4)   Redução de erros
Colunas com tipos corretos diminuem a probabilidade de falhas durante o treinamento. Erros nos dados podem levar a padrões incorretos e resultados imprecisos.

**Resumo:**
Converter colunas para seus tipos corretos é uma etapa essencial da preparação de dados, garantindo que o modelo de machine learning seja preciso, eficiente e confiável.

##Identificando e tratando strings vazias

Visualizaremos o conteúdo presente em cada coluna de dados_normalizados. Para isso, executaremos o comando abaixo, onde há um for que percorrerá cada nome, dentro do qual fazemos três prints que passam, respectivamente:

*   Uma f-string para visualizar o nome da coluna{col} a cada repetição do laço
*   Uma referência da coluna visualizada junto aos valores únicos presentes nela
*   Um "-" * 30 para exibir 30 traços para organização do retorno

In [12]:
for col in dados_normalizados.columns:
    print(f"Coluna: {col}")
    print(dados_normalizados[col].unique())
    print("-" * 30)

Coluna: id_cliente
['0002-ORFBO' '0003-MKNFE' '0004-TLHLJ' ... '9992-UJOEL' '9993-LHIEB'
 '9995-HOTOH']
------------------------------
Coluna: Churn
['nao' 'sim' '']
------------------------------
Coluna: cliente.genero
['feminino' 'masculino']
------------------------------
Coluna: cliente.idoso
[0 1]
------------------------------
Coluna: cliente.parceiro
['sim' 'nao']
------------------------------
Coluna: cliente.dependentes
['sim' 'nao']
------------------------------
Coluna: cliente.tempo_servico
[9.00e+00 4.00e+00 1.30e+01 3.00e+00 7.10e+01 6.30e+01 7.00e+00      nan
 5.40e+01 7.20e+01 5.00e+00 5.60e+01 3.40e+01 1.00e+00 4.50e+01 5.00e+01
 2.30e+01 5.50e+01 2.60e+01 6.90e+01 1.10e+01 3.70e+01 4.90e+01 6.60e+01
 6.70e+01 2.00e+01 4.30e+01 5.90e+01 1.20e+01 2.70e+01 2.00e+00 2.50e+01
 2.90e+01 1.40e+01 3.50e+01 6.40e+01 3.90e+01 4.00e+01 6.00e+00 3.00e+01
 7.00e+01 5.70e+01 5.80e+01 1.60e+01 3.20e+01 3.30e+01 1.00e+01 2.10e+01
 6.10e+01 1.50e+01 4.40e+01 2.20e+01 2.40e+01 1.90e+01

Na coluna Churn, além de "não" e "sim", temos um valor estranho: ''. Vamos investigá-lo.

`query`

In [15]:
# Seleciona linhas onde a coluna 'Churn' contém strings vazias
dados_normalizados.query("Churn == ''")

,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
30,0047-ZHDTW,,feminino,0,nao,nao,11.0,sim,sim,fibra otica,...,nao,nao,nao,nao,nao,mes a mes,sim,transferencia bancaria (automatica),79.00,929.30
75,0120-YZLQA,,masculino,0,nao,nao,71.0,sim,nao,nao,...,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,dois anos,sim,cartao de credito (automatico),19.90,1355.10
96,0154-QYHJU,,masculino,0,nao,nao,29.0,sim,nao,DSL,...,sim,nao,sim,nao,nao,um ano,sim,cheque eletronico,58.75,1696.20
98,0162-RZGMZ,,feminino,1,nao,nao,5.0,sim,nao,DSL,...,sim,nao,sim,nao,nao,mes a mes,nao,cartao de credito (automatico),59.90,287.85
175,0274-VVQOQ,,masculino,1,sim,nao,65.0,sim,sim,fibra otica,...,sim,sim,nao,sim,sim,um ano,sim,transferencia bancaria (automatica),103.15,6792.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7211,9920-GNDMB,,masculino,0,nao,nao,9.0,sim,sim,fibra otica,...,nao,nao,nao,nao,nao,mes a mes,sim,cheque eletronico,76.25,684.85
7239,9955-RVWSC,,feminino,0,sim,sim,67.0,sim,nao,nao,...,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,dois anos,sim,transferencia bancaria (automatica),19.25,1372.90
7247,9966-VYRTZ,,feminino,0,sim,sim,31.0,sim,nao,nao,...,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,mes a mes,sim,cheque pelo correio,19.55,658.95
7267,6532-YOHZY,,masculino,0,sim,sim,45.0,sim,sim,fibra otica,...,sim,sim,sim,sim,sim,dois anos,sim,transferencia bancaria (automatica),109.75,4900.65


##Descartando amostras

In [14]:
dados_normalizados[dados_normalizados['Churn'] != '']

,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,internet.servico_internet,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
0,0002-ORFBO,nao,feminino,0,sim,sim,9.0,sim,nao,DSL,...,sim,nao,sim,sim,nao,None,None,None,NaN,NaN
1,0003-MKNFE,nao,masculino,0,nao,nao,9.0,sim,sim,DSL,...,nao,nao,nao,nao,sim,mes a mes,nao,cheque pelo correio,59.90,542.40
2,0004-TLHLJ,sim,masculino,0,nao,nao,4.0,sim,nao,fibra otica,...,nao,sim,nao,nao,nao,mes a mes,sim,cheque eletronico,73.90,280.85
3,0011-IGKFF,sim,masculino,1,sim,nao,13.0,sim,nao,fibra otica,...,sim,sim,nao,sim,sim,mes a mes,sim,cheque eletronico,98.00,1237.85
4,0013-EXCHZ,sim,feminino,1,sim,nao,3.0,sim,nao,fibra otica,...,nao,nao,sim,sim,nao,mes a mes,sim,cheque pelo correio,83.90,267.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7338,5792-JALQC,nao,feminino,1,nao,nao,52.0,sim,sim,DSL,...,nao,sim,nao,nao,nao,dois anos,nao,transferencia bancaria (automatica),59.85,3103.25
7339,5172-RKOCB,nao,masculino,0,sim,nao,72.0,sim,sim,fibra otica,...,sim,nao,sim,sim,sim,dois anos,sim,cartao de credito (automatico),108.95,7875.00
7340,1934-MKPXS,nao,masculino,0,sim,sim,33.0,sim,nao,nao,...,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,um ano,nao,cartao de credito (automatico),20.10,620.55
7341,5959-BELXA,sim,masculino,1,nao,nao,32.0,sim,sim,fibra otica,...,nao,nao,nao,sim,sim,mes a mes,sim,cartao de credito (automatico),96.15,3019.25


In [16]:
dados_sem_vazio = dados_normalizados[dados_normalizados['Churn'] != '']

##Criando uma cópia do dataframe


Para criar uma cópia independente, adicionaremos no final do comando anterior um .copy().

`copy()`

In [17]:
dados_sem_vazio = dados_normalizados[dados_normalizados['Churn'] != ''].copy()

In [18]:
dados_sem_vazio.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7118 entries, 0 to 7343
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id_cliente                     7118 non-null   object 
 1   Churn                          7118 non-null   object 
 2   cliente.genero                 7118 non-null   object 
 3   cliente.idoso                  7118 non-null   int64  
 4   cliente.parceiro               7118 non-null   object 
 5   cliente.dependentes            7118 non-null   object 
 6   cliente.tempo_servico          7110 non-null   float64
 7   telefone.servico_telefone      7118 non-null   object 
 8   telefone.varias_linhas         7118 non-null   object 
 9   internet.servico_internet      7118 non-null   object 
 10  internet.seguranca_online      7118 non-null   object 
 11  internet.backup_online         7118 non-null   object 
 12  internet.protecao_dispositivo  7118 non-null   object

No retorno, veremos um Int64Index informando que existem 7118 entradas, cujos índices vão de 0 a 7343 — ou seja, existem índices saltados.

##Corrigindo o índice das amostras

In [19]:
dados_sem_vazio.reset_index()

,index,id_cliente,Churn,cliente.genero,cliente.idoso,cliente.parceiro,cliente.dependentes,cliente.tempo_servico,telefone.servico_telefone,telefone.varias_linhas,...,internet.backup_online,internet.protecao_dispositivo,internet.suporte_tecnico,internet.tv_streaming,internet.filmes_streaming,conta.contrato,conta.faturamente_eletronico,conta.metodo_pagamento,conta.cobranca.mensal,conta.cobranca.Total
0,0,0002-ORFBO,nao,feminino,0,sim,sim,9.0,sim,nao,...,sim,nao,sim,sim,nao,None,None,None,NaN,NaN
1,1,0003-MKNFE,nao,masculino,0,nao,nao,9.0,sim,sim,...,nao,nao,nao,nao,sim,mes a mes,nao,cheque pelo correio,59.90,542.40
2,2,0004-TLHLJ,sim,masculino,0,nao,nao,4.0,sim,nao,...,nao,sim,nao,nao,nao,mes a mes,sim,cheque eletronico,73.90,280.85
3,3,0011-IGKFF,sim,masculino,1,sim,nao,13.0,sim,nao,...,sim,sim,nao,sim,sim,mes a mes,sim,cheque eletronico,98.00,1237.85
4,4,0013-EXCHZ,sim,feminino,1,sim,nao,3.0,sim,nao,...,nao,nao,sim,sim,nao,mes a mes,sim,cheque pelo correio,83.90,267.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7113,7338,5792-JALQC,nao,feminino,1,nao,nao,52.0,sim,sim,...,nao,sim,nao,nao,nao,dois anos,nao,transferencia bancaria (automatica),59.85,3103.25
7114,7339,5172-RKOCB,nao,masculino,0,sim,nao,72.0,sim,sim,...,sim,nao,sim,sim,sim,dois anos,sim,cartao de credito (automatico),108.95,7875.00
7115,7340,1934-MKPXS,nao,masculino,0,sim,sim,33.0,sim,nao,...,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,sem servico de internet,um ano,nao,cartao de credito (automatico),20.10,620.55
7116,7341,5959-BELXA,sim,masculino,1,nao,nao,32.0,sim,sim,...,nao,nao,nao,sim,sim,mes a mes,sim,cartao de credito (automatico),96.15,3019.25


Já que não queremos os índices antigos, adicionaremos entre os parênteses do reset_index():

*  Um drop=True que retirará a coluna Index
*  Um inplace=True que alterará diretamente o dados_sem_vazio

In [20]:
dados_sem_vazio.reset_index(drop=True, inplace=True)

#Exercício: tratando os dados

In [32]:
dados = {
  "pessoas": [
    {
      "nome": "João",
      "idade": "25",
      "endereco": {
        "rua": "Rua A",
        "numero": 123,
        "cidade": "São Paulo"
      },
      "telefones": [
        "11 1111-1111",
        "11 2222-2222"
      ]
    },
    {
      "nome": "Maria",
      "idade": 30,
      "endereco": {
        "rua": "",
        "numero": 456,
        "cidade": "Rio de Janeiro"
      },
      "telefones": [
        "21 3333-3333"
      ]
    }
  ]
}
import json

with open('informacoes.json', 'w') as f:
    json.dump(dados, f)

print("Dados salvos em informacoes.json")

Dados salvos em informacoes.json


In [25]:
dados

{'pessoas': [{'nome': 'João',
   'idade': '25',
   'endereco': {'rua': 'Rua A', 'numero': 123, 'cidade': 'São Paulo'},
   'telefones': ['11 1111-1111', '11 2222-2222']},
  {'nome': 'Maria',
   'idade': 30,
   'endereco': {'rua': '', 'numero': 456, 'cidade': 'Rio de Janeiro'},
   'telefones': ['21 3333-3333']}]}

In [35]:
df = pd.json_normalize(dados, record_path= ['pessoas'])
df

,nome,idade,telefones,endereco.rua,endereco.numero,endereco.cidade
0,João,25,"[11 1111-1111, 11 2222-2222]",Rua A,123,São Paulo
1,Maria,30,[21 3333-3333],,456,Rio de Janeiro


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   nome             2 non-null      object
 1   idade            2 non-null      object
 2   telefones        2 non-null      object
 3   endereco.rua     2 non-null      object
 4   endereco.numero  2 non-null      int64 
 5   endereco.cidade  2 non-null      object
dtypes: int64(1), object(5)
memory usage: 228.0+ bytes


In [39]:
df['idade'] = df['idade'].astype(int)
df_filtrado = df.query('`endereco.rua` != ""')

In [41]:
df_filtrado = df[df['endereco.rua'] != ""]
df_filtrado

,nome,idade,telefones,endereco.rua,endereco.numero,endereco.cidade
0,João,25,"[11 1111-1111, 11 2222-2222]",Rua A,123,São Paulo
